# DeepSTARR for multi-task enhancer design

[Notebook](multi-task.ipynb) · [Installation](../docs/installation.md)

Increase predicted housekeeping activity and decrease developmental activity in 249-nt enhancers.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import MultiTaskObjective, call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model("deepstarr", device=device)
objective = MultiTaskObjective(high=["housekeeping"], low=["developmental"])
seeds = random_seed_sequences(249, 100, seed=42)
run = Path("outputs/deepstarr")




## 2. Generate candidates



In [2]:
designs = generate_designs(
    model, length=249, initial=seeds, objective=objective,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,1.650070,1.705994
basesearch,100,1.637914,1.712034
beam_search,100,1.832773,1.986408
cross_entropy_method,100,1.761767,1.832947
den,100,1.671228,1.752437
fast_seqprop,100,1.756840,1.840970
genetic_algorithm,100,1.743714,1.791901
ism_greedy,100,1.587428,1.792361
ledidi,100,1.442902,1.653876




## 3. Discover motifs



In [3]:
motifs = call_motifs(designs, model, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-AYTATCGATART,758,635,1333.0
1,3-AGTGTTGS,142,135,177.0
2,4-YGGTCACAC,105,104,46.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,TCGCCTTTTGGGTCCTGTTTCCCGGGTGTCCCTGAAGTCCTGTGGG...,1.986408,1-AYTATCGATART
471,2,AATTCTGTGGTGTAGGACCAGGTGGACCGACTACTCCTCATGTAAC...,1.696594,4-YGGTCACAC
102,3,CTGCCTCCTTGGGCTTCGGAGGGGGGGGGGTCGGGATGAGCGAGAT...,1.795960,3-AGTGTTGS
526,4,ATGGTGCAACCACTAAGCTCGAATTAACCGCTATTCGATAACACCG...,1.679834,1-AYTATCGATART;4-YGGTCACAC
138,5,ACCTATCAAATAAGGTTGGAAAATAGTATTGAAACATCGCAAACGA...,1.783513,1-AYTATCGATART;3-AGTGTTGS
862,6,ATAGAAATATAACGCGTAGTTAGCAGTGTTGGGCCGCTTCGAAAAT...,1.566787,3-AGTGTTGS
807,7,AGTACTGAATTCCTAATATCCATAAGGTAAGTCTTTGGGGGGCGTT...,1.622481,4-YGGTCACAC
571,8,TGGGCGATTCCCTAAGACAGGTTAAGAGTCGATATTATCGAGTTAT...,1.669706,3-AGTGTTGS
683,9,CCGTCTGCACGATCTGCAGGATCACCTGAAGACAATAATATAAAGG...,1.652395,1-AYTATCGATART
27,10,GTCCTGCGTCCACCGCGGGCAACAGTTCGAGGGTCTCGACCGTTAC...,1.892810,1-AYTATCGATART
